In [ ]:
!wget http://www.vision.caltech.edu/Image_Datasets/Caltech101/101_ObjectCategories.tar.gz
!tar -xvf 101_ObjectCategories.tar.gz

--2026-03-25 14:21:29--  http://www.vision.caltech.edu/Image_Datasets/Caltech101/101_ObjectCategories.tar.gz
Resolving www.vision.caltech.edu (www.vision.caltech.edu)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to www.vision.caltech.edu (www.vision.caltech.edu)|185.199.108.153|:80... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-25 14:21:29 ERROR 404: Not Found.

tar: 101_ObjectCategories.tar.gz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now


In [1]:
!wget http://www.vision.caltech.edu/Image_Datasets/Caltech101/101_ObjectCategories.tar.gz
!tar -xvf 101_ObjectCategories.tar.gz

--2026-03-28 17:33:55--  http://www.vision.caltech.edu/Image_Datasets/Caltech101/101_ObjectCategories.tar.gz
Resolving www.vision.caltech.edu (www.vision.caltech.edu)... 185.199.109.153, 185.199.110.153, 185.199.108.153, ...
Connecting to www.vision.caltech.edu (www.vision.caltech.edu)|185.199.109.153|:80... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-28 17:33:55 ERROR 404: Not Found.

tar: 101_ObjectCategories.tar.gz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now


In [2]:
!pip install tensorflow-datasets

In [4]:
import tensorflow_datasets as tfds

# Load dataset
dataset, info = tfds.load('caltech101', split='train', with_info=True)

print(info)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/caltech101/incomplete.LWK8C0_3.0.2/caltech101-train.tfrecord*...:   0%|   …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/caltech101/incomplete.LWK8C0_3.0.2/caltech101-test.tfrecord*...:   0%|    …

Dataset caltech101 downloaded and prepared to /root/tensorflow_datasets/caltech101/3.0.2. Subsequent calls will reuse this data.
tfds.core.DatasetInfo(
    name='caltech101',
    full_name='caltech101/3.0.2',
    description="""
    Caltech-101 consists of pictures of objects belonging to 101 classes, plus one
    `background clutter` class. Each image is labelled with a single object. Each
    class contains roughly 40 to 800 images, totalling around 9k images. Images are
    of variable sizes, with typical edge lengths of 200-300 pixels. This version
    contains image-level labels only. The original dataset also contains bounding
    boxes.
    """,
    homepage='https://doi.org/10.22002/D1.20086',
    data_dir='/root/tensorflow_datasets/caltech101/3.0.2',
    file_format=tfrecord,
    download_size=131.05 MiB,
    dataset_size=132.86 MiB,
    features=FeaturesDict({
        'image': Image(shape=(None, None, 3), dtype=uint8),
        'image/file_name': Text(shape=(), dtype=string),


In [5]:
import numpy as np
import cv2

images = []
labels = []

# Take limited samples (important)
for i, data in enumerate(tfds.as_numpy(dataset)):

    if i >= 400:  # limit dataset size
        break

    img = data['image']
    label = data['label']

    # Resize + grayscale
    img = cv2.resize(img, (64, 64))
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    images.append(img / 255.0)
    labels.append(label)

images = np.array(images)
labels = np.array(labels)

print(images.shape)

(400, 64, 64)


In [6]:
# Feature extraction
features = [extract_lbp_features(img) for img in images]
features = np.array(features)

# Evaluation
precision, recall, mAP = evaluate_system(images, labels, features)

print_evaluation_table(precision, recall, mAP)

NameError: name 'extract_lbp_features' is not defined

In [7]:
import numpy as np
from skimage.feature import local_binary_pattern
from sklearn.metrics.pairwise import euclidean_distances

# -------------------------------
# LBP Feature Extraction
# -------------------------------
def extract_lbp_features(image):
    radius = 1
    n_points = 8 * radius

    lbp = local_binary_pattern(image, n_points, radius, method='uniform')

    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_points + 3),
        range=(0, n_points + 2)
    )

    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    return hist


# -------------------------------
# Retrieval Function
# -------------------------------
def retrieve_similar_images(query_img, features, top_k=5):

    query_feat = extract_lbp_features(query_img)

    dists = euclidean_distances([query_feat], features)[0]

    indices = np.argsort(dists)[:top_k]

    return indices


# -------------------------------
# Precision & Recall
# -------------------------------
def compute_precision_recall(query_label, retrieved_indices, labels):

    relevant = 0
    for idx in retrieved_indices:
        if labels[idx] == query_label:
            relevant += 1

    precision = relevant / len(retrieved_indices)

    total_relevant = np.sum(labels == query_label)
    recall = relevant / total_relevant

    return precision, recall


# -------------------------------
# Average Precision (AP)
# -------------------------------
def compute_ap(query_label, retrieved_indices, labels):

    correct = 0
    precision_sum = 0

    for i, idx in enumerate(retrieved_indices):
        if labels[idx] == query_label:
            correct += 1
            precision_at_i = correct / (i + 1)
            precision_sum += precision_at_i

    if correct == 0:
        return 0

    return precision_sum / correct


# -------------------------------
# Evaluation
# -------------------------------
def evaluate_system(x_data, labels, features, num_queries=100, top_k=5):

    precisions = []
    recalls = []
    aps = []

    for i in range(num_queries):

        query_img = x_data[i]
        query_label = labels[i]

        retrieved = retrieve_similar_images(query_img, features, top_k)

        p, r = compute_precision_recall(query_label, retrieved, labels)
        ap = compute_ap(query_label, retrieved, labels)

        precisions.append(p)
        recalls.append(r)
        aps.append(ap)

    return np.mean(precisions), np.mean(recalls), np.mean(aps)


# -------------------------------
# Print Table
# -------------------------------
def print_evaluation_table(precision, recall, mAP):

    print("\nComparative Analysis\n")

    print("{:<10} {:<10} {:<10} {:<10} {:<15} {:<40}".format(
        "Method", "Precision", "Recall", "mAP", "Robustness", "Remarks"
    ))

    print("-" * 100)

    print("{:<10} {:<10.2f} {:<10.2f} {:<10.2f} {:<15} {:<40}".format(
        "LBP",
        precision,
        recall,
        mAP,
        "Low",
        "Texture-based, limited semantic understanding"
    ))

In [8]:
# Feature extraction
features = [extract_lbp_features(img) for img in images]
features = np.array(features)

# Evaluation
precision, recall, mAP = evaluate_system(images, labels, features)

print_evaluation_table(precision, recall, mAP)

/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
/usr/local/lib/python3


Comparative Analysis

Method     Precision  Recall     mAP        Robustness      Remarks                                 
----------------------------------------------------------------------------------------------------
LBP        0.25       0.30       0.97       Low             Texture-based, limited semantic understanding


/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
/usr/local/lib/python3